In [1]:
import os
import json

import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import entropy, kurtosis, skew
from collections import defaultdict

from src.dataset.dataset_info import datasets
from src.numpy_encoder import NumpyEncoder
from src.graph.graph_measures import calculate_graph_measures
from local_variables import local_datasets_path


In [2]:
save_to_file = True

In [3]:
my_datasets = [
    # datasets["cic_ids_2017_5_percent"],
    datasets["cic_ton_iot"],
    datasets["cic_ids_2017"],
    datasets["cic_bot_iot"],
    datasets["cic_ton_iot_modified"],
    datasets["ccd_inid_modified"],
    datasets["nf_uq_nids_modified"],
    datasets["edge_iiot"],
    datasets["nf_cse_cic_ids2018"],
    datasets["nf_uq_nids"],
    datasets["x_iiot"],
]

In [4]:
results = {}
new_file_name = "df_properties_new.json"

In [5]:
def connected_to_attackers(graph, label_col):

    # Step 1: Identify attacker nodes
    attackers = set()
    victims = set()
    for u, v, data in graph.edges(data=True):
        if data[label_col] == 1:
            attackers.add(u)
            victims.add(v)

    # Step 2: Count unique attackers and victims
    num_attackers = len(attackers)
    num_victims = len(victims)

    # Step 3: Calculate proportions
    total_nodes = graph.number_of_nodes()
    attacker_proportion = num_attackers / total_nodes if total_nodes > 0 else 0
    victim_proportion = num_victims / total_nodes if total_nodes > 0 else 0

    # Step 4: Find how many non-attackers are connected to attackers
    connected_to_attackers = set()
    for attacker in attackers:
        for neighbor in graph.neighbors(attacker):
            if neighbor not in attackers:  # Ensure the neighbor is not also an attacker
                connected_to_attackers.add(neighbor)

    # Step 5: Compute proportion of non-attackers connected to attackers
    num_connected_to_attackers = len(connected_to_attackers)
    connected_proportion = num_connected_to_attackers / total_nodes if total_nodes > 0 else 0

    # Output results
    print(f"Number of attackers: {num_attackers}")
    print(f"Number of victims: {num_victims}")
    print(f"Proportion of attackers: {attacker_proportion:.4f}")
    print(f"Proportion of victims: {victim_proportion:.4f}")
    print(f"Number of non-attackers connected to attackers: {num_connected_to_attackers}")
    print(f"Proportion of non-attackers connected to attackers: {connected_proportion:.4f}")


In [6]:
def get_edge_class_distribution(graph, label_col):
    """
    Computes the distribution of edge classes connected to each node.
    
    Args:
        graph: A NetworkX MultiGraph or MultiDiGraph.
        label_col: The key in edge attributes that stores the class label.

    Returns:
        node_edge_class_counts: A dictionary where keys are nodes, and values are dictionaries
                                mapping edge class to its count.
    """
    node_edge_class_counts = defaultdict(lambda: defaultdict(int))

    for u, v, data in graph.edges(data=True):
        edge_class = data[label_col]  # Get the class of the edge
        node_edge_class_counts[u][edge_class] += 1
        node_edge_class_counts[v][edge_class] += 1  # Since edges are undirected, count for both nodes

    return node_edge_class_counts

In [7]:
# for g in graphs_types:
#     if not g.with_ports:
#         print(g.name)
#         node_class_distribution = get_edge_class_distribution(g.graph, dataset.class_num_col)
#         for node, class_counts in list(node_class_distribution.items())[:10]:  # Show first 10 nodes
#             print(f"Node {node}: {dict(class_counts)}")

In [8]:
def compute_edge_class_entropy(graph, label_col):
    """
    Computes the average entropy of edge class distributions across all nodes.
    
    Args:
        graph: A NetworkX MultiGraph or MultiDiGraph.
        label_col: The key in edge attributes that stores the class label.

    Returns:
        avg_entropy: The average entropy of all nodes.
    """
    node_entropy = {}

    for node in graph.nodes():
        edge_class_counts = defaultdict(int)
        total_edges = 0

        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)

            # If graph is a MultiGraph, edge_data is a dictionary with multiple edges
            if isinstance(edge_data, dict):  
                for edge_id, data in edge_data.items():
                    edge_class = data.get(label_col, None)
                    if edge_class is not None:
                        edge_class_counts[edge_class] += 1
                        total_edges += 1
            else:  # If it's a simple Graph/DiGraph, just process normally
                edge_class = edge_data.get(label_col, None)
                if edge_class is not None:
                    edge_class_counts[edge_class] += 1
                    total_edges += 1

        # Compute entropy for the node
        if total_edges > 0:
            probs = np.array(list(edge_class_counts.values())) / total_edges
            entropy = -np.sum(probs * np.log2(probs))  # Shannon entropy
        else:
            entropy = 0  # If node has no edges, entropy is 0

        node_entropy[node] = entropy

    # Compute average entropy across all nodes
    avg_entropy = np.mean(list(node_entropy.values())) if node_entropy else 0
    return avg_entropy


# Higher entropy → The node interacts with diverse attack/traffic types.
# Lower entropy → The node interacts with one dominant edge class.
# Zero entropy → Either an isolated node or all edges belong to the same class.


In [9]:
def compute_avg_edge_class_diversity(graph, label_col):
    """
    Computes the average number of unique edge classes per node.
    
    Args:
        graph: A NetworkX MultiGraph or MultiDiGraph.
        label_col: The key in edge attributes that stores the class label.

    Returns:
        avg_diversity: Average unique edge class count per node.
    """
    node_diversity = []

    for node in graph.nodes():
        unique_classes = set()
        total_edges = 0

        for neighbor in graph.neighbors(node):
            edge_data = graph.get_edge_data(node, neighbor)

            # If MultiGraph, iterate through multiple edges
            if isinstance(edge_data, dict):  
                for edge_id, data in edge_data.items():
                    edge_class = data.get(label_col, None)
                    if edge_class is not None:
                        unique_classes.add(edge_class)
                        total_edges += 1
            else:  # If it's a simple Graph, process normally
                edge_class = edge_data.get(label_col, None)
                if edge_class is not None:
                    unique_classes.add(edge_class)
                    total_edges += 1

        # Compute diversity for the node
        if total_edges > 0:
            diversity = len(unique_classes) / total_edges
            node_diversity.append(diversity)

    return np.mean(node_diversity) if node_diversity else 0

# Higher diversity → The node interacts with multiple types of traffic.
# Lower diversity → The node interacts mostly with one edge class.
# Zero diversity → The node has no edges or only interacts with one type.

In [10]:
# for g in graphs_types:
#     if not g.with_ports:
#         print(g.name)
#         average_diversity = compute_avg_edge_class_diversity(g.graph, dataset.class_num_col)
#         print(f"Average Edge Class Diversity (Unique Class Ratio): {average_diversity:.4f}")

In [11]:
for dataset in my_datasets:
    print("=========================")
    print("=========================")
    print(f"==>> dataset: {dataset.name}")
    
    folder_path = os.path.join(local_datasets_path,dataset.name)
    df = pd.read_parquet(os.path.join(folder_path, f"{dataset.name}.parquet"))
    G = nx.from_pandas_edgelist(df, dataset.src_ip_col, dataset.dst_ip_col, edge_attr=[dataset.label_col, dataset.class_num_col], create_using=nx.MultiDiGraph())
    connected_to_attackers(G, dataset.label_col)
    
    average_entropy = compute_edge_class_entropy(G, dataset.class_num_col)
    print(f"Average Edge Class Diversity (Entropy): {average_entropy:.4f}")
    
    average_diversity = compute_avg_edge_class_diversity(G, dataset.class_num_col)
    print(f"Average Edge Class Diversity (Unique Class Ratio): {average_diversity:.4f}")

==>> dataset: cic_ton_iot
Number of attackers: 11
Number of victims: 116
Proportion of attackers: 0.0001
Proportion of victims: 0.0008
Number of non-attackers connected to attackers: 243
Proportion of non-attackers connected to attackers: 0.0017
Average Edge Class Diversity (Entropy): 0.0001
Average Edge Class Diversity (Unique Class Ratio): 0.5253
==>> dataset: cic_ids_2017
Number of attackers: 10
Number of victims: 11
Proportion of attackers: 0.0005
Proportion of victims: 0.0006
Number of non-attackers connected to attackers: 16500
Proportion of non-attackers connected to attackers: 0.8626
Average Edge Class Diversity (Entropy): 0.0001
Average Edge Class Diversity (Unique Class Ratio): 0.3379
==>> dataset: cic_bot_iot
Number of attackers: 8
Number of victims: 8
Proportion of attackers: 0.0272
Proportion of victims: 0.0272
Number of non-attackers connected to attackers: 275
Proportion of non-attackers connected to attackers: 0.9354
Average Edge Class Diversity (Entropy): 0.0329
Averag